In [1]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import subprocess
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)

print(f"🚀 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Ensure we are in the root directory for relative paths
while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"📁 Working directory set to: {os.getcwd()}")

print("✅ Environment ready!")

🚀 Environment: Local
📁 Working directory set to: /home/aidmantas/repos/computer-data-analysis-report
✅ Environment ready!


# 04 - Quarterly Panel Dataset

**Goal**: Build a quarterly panel dataset (Company × Quarter) to enable statistically meaningful ML on promise outcomes.

**Output**: `data/processed/quarterly_panel.csv` (~500 rows × 25 columns)

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Create output directory
os.makedirs('data/processed', exist_ok=True)
print("✅ Libraries imported, output directory ready")

✅ Libraries imported, output directory ready


## Step 1: Load All Data Sources

In [3]:
# Load company financials (daily)
df_fin = pd.read_csv('data/raw/company_financials_expanded.csv')
df_fin['Date'] = pd.to_datetime(df_fin['Date'], format='mixed', utc=True).dt.tz_localize(None)
print(f"📈 Company financials: {len(df_fin):,} rows, {df_fin['ticker'].nunique()} tickers")
print(f"   Date range: {df_fin['Date'].min()} to {df_fin['Date'].max()}")

# Load financial ratios (static)
df_ratios = pd.read_csv('data/raw/company_financial_ratios.csv')
print(f"📊 Financial ratios: {len(df_ratios)} companies")

# Load macro indicators
df_macro = pd.read_csv('data/raw/macro_economic_indicators.csv')
df_macro['date'] = pd.to_datetime(df_macro['date'], format='mixed', utc=True).dt.tz_localize(None)
print(f"🏦 Macro indicators: {len(df_macro):,} rows")

# Load grid interconnection queue
df_grid = pd.read_csv('data/raw/grid_interconnection_queue.csv')
df_grid['Queue Date'] = pd.to_datetime(df_grid['Queue Date'], format='mixed', utc=True).dt.tz_localize(None)
print(f"⚡ Grid queue: {len(df_grid):,} projects")

# Load buildout promises (labels)
df_promises = pd.read_csv('data/raw/buildout_promises_expanded.csv')
df_promises['announcement_date'] = pd.to_datetime(df_promises['announcement_date'], format='mixed', utc=True).dt.tz_localize(None)
df_promises['target_date'] = pd.to_datetime(df_promises['target_date'], format='mixed', utc=True).dt.tz_localize(None)
print(f"🤝 Buildout promises: {len(df_promises)} promise events")

📈 Company financials: 13,816 rows, 11 tickers
   Date range: 2021-04-21 04:00:00 to 2026-04-21 04:00:00
📊 Financial ratios: 19 companies
🏦 Macro indicators: 1,665 rows
⚡ Grid queue: 6,043 projects
🤝 Buildout promises: 34 promise events


## Step 2: Create Quarterly Keys

In [4]:
# Add quarter column to financials
df_fin['quarter'] = df_fin['Date'].dt.to_period('Q')
df_fin['year'] = df_fin['Date'].dt.year
df_fin['quarter_num'] = df_fin['Date'].dt.quarter

# Get unique tickers
tickers = df_fin['ticker'].unique()
print(f"📌 Unique tickers: {list(tickers)}")

# Create all (ticker, quarter) combinations
quarters = df_fin['quarter'].unique()
quarters = sorted(quarters)
print(f"📌 Date range: {quarters[0]} to {quarters[-1]} ({len(quarters)} quarters)")

# Build complete panel
panel_data = []
for ticker in tickers:
    for q in quarters:
        panel_data.append({'ticker': ticker, 'quarter': str(q)})

df_panel = pd.DataFrame(panel_data)
print(f"📋 Initial panel: {len(df_panel)} rows ({len(tickers)} companies × {len(quarters)} quarters)")

📌 Unique tickers: ['DLR', 'EQIX', 'AMT', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'IRM', 'SBAC', 'CCI']
📌 Date range: 2021Q2 to 2026Q2 (21 quarters)
📋 Initial panel: 231 rows (11 companies × 21 quarters)


## Step 3: Aggregate Financial Features by Quarter

In [5]:
# Aggregate financial metrics per (ticker, quarter)
agg_fin = df_fin.groupby(['ticker', 'quarter']).agg({
    'Close': ['mean', 'std', 'min', 'max'],
    'Volume': ['mean', 'sum'],
    'Open': 'mean',
    'High': 'mean',
    'Low': 'mean'
}).reset_index()

# Flatten column names
agg_fin.columns = ['ticker', 'quarter', 
                   'q_avg_close', 'q_volatility', 'q_min_close', 'q_max_close',
                   'q_avg_volume', 'q_total_volume',
                   'q_avg_open', 'q_avg_high', 'q_avg_low']

# Fill NaN volatility (single-day quarters) with 0
agg_fin['q_volatility'] = agg_fin['q_volatility'].fillna(0)

print(f"📊 Aggregated financials: {len(agg_fin)} rows")
print(agg_fin.head())

📊 Aggregated financials: 231 rows
  ticker quarter  q_avg_close  q_volatility  q_min_close  q_max_close  \
0    AMT  2021Q2   221.560922      8.271164   207.934204   235.551102   
1    AMT  2021Q3   246.214487      7.088921   230.247452   262.172424   
2    AMT  2021Q4   237.114648      8.307338   223.411407   255.008789   
3    AMT  2022Q1   212.736932     11.120704   197.790878   249.673218   
4    AMT  2022Q2   220.002206     10.882699   196.963715   234.538605   

   q_avg_volume  q_total_volume  q_avg_open  q_avg_high   q_avg_low  
0  1.779350e+06        88967500  221.286304  223.010887  219.565107  
1  1.418830e+06        90805100  246.210824  248.078275  244.042672  
2  1.649081e+06       105541200  236.986296  239.352545  234.631681  
3  2.089189e+06       129529700  213.231623  215.611141  210.044539  
4  2.140958e+06       132739400  219.561257  222.951722  216.650772  


## Step 4: Join Static Financial Ratios

In [6]:
# Select static ratio columns
ratio_cols = ['ticker', 'market_cap', 'enterprise_value', 'revenue', 'ebitda',
              'profit_margin', 'operating_margin', 'return_on_assets', 'return_on_equity',
              'debt_to_equity', 'current_ratio', 'quick_ratio', 'dividend_yield', 
              'beta', 'forward_pe', 'trailing_pe', 'price_to_book', 'price_to_sales']

df_ratios_subset = df_ratios[ratio_cols].copy()

# Join to panel
df_panel = df_panel.merge(df_ratios_subset, on='ticker', how='left')
print(f"✅ Joined ratios: {len(df_panel)} rows")
print(f"   Missing ratios: {df_panel['profit_margin'].isna().sum()}")

✅ Joined ratios: 231 rows
   Missing ratios: 0


## Step 5: Resample Macro Indicators to Quarterly

In [7]:
# Add quarter to macro data
df_macro['quarter'] = df_macro['date'].dt.to_period('Q')

# Resample macro to quarterly (forward-fill GDP/CPI, use mean for rates)
macro_quarterly = df_macro.groupby('quarter').agg({
    'Real GDP': 'last',  # GDP is quarterly anyway
    'Core CPI': 'last',
    'Fed Funds Rate': 'mean',
    'Unemployment Rate': 'mean',
    '30Y Mortgage Rate': 'mean',
    '10Y Treasury': 'mean'
}).reset_index()

# Forward fill missing values
macro_quarterly = macro_quarterly.ffill()
macro_quarterly['quarter'] = macro_quarterly['quarter'].astype(str)

print(f"🏦 Quarterly macro: {len(macro_quarterly)} quarters")
print(macro_quarterly.head())

🏦 Quarterly macro: 26 quarters
  quarter   Real GDP  Core CPI  Fed Funds Rate  Unemployment Rate  \
0  2020Q1  20709.212   267.054        1.260000           3.833333   
1  2020Q2  19077.992   265.849        0.060000          13.000000   
2  2020Q3  20558.879   268.933        0.093333           8.800000   
3  2020Q4  20791.917   270.341        0.090000           6.766667   
4  2021Q1  21082.134   271.459        0.080000           6.233333   

   30Y Mortgage Rate  10Y Treasury  
0           3.521538      1.365000  
1           3.239231      0.687619  
2           2.952308      0.650625  
3           2.760714      0.864516  
4           2.875833      1.335902  


In [8]:
# Join macro to panel
df_panel = df_panel.merge(macro_quarterly, on='quarter', how='left')
print(f"✅ Joined macro: {len(df_panel)} rows")

✅ Joined macro: 231 rows


## Step 6: Aggregate Grid Queue Depth by Quarter

In [9]:
# Add quarter to grid queue
df_grid['quarter'] = df_grid['Queue Date'].dt.to_period('Q')

# Count active projects per ISO per quarter
grid_quarterly = df_grid.groupby(['iso', 'quarter']).size().reset_index(name='iso_queue_depth')
grid_quarterly['iso'] = grid_quarterly['iso'].str.upper()  # Standardize
grid_quarterly['quarter'] = grid_quarterly['quarter'].astype(str)

# Pivot to wide format (one column per ISO)
grid_pivot = grid_quarterly.pivot(index='quarter', columns='iso', values='iso_queue_depth').reset_index()
grid_pivot.columns = ['quarter'] + [f'queue_{col}' for col in grid_pivot.columns[1:]]
grid_pivot = grid_pivot.fillna(0)

print(f"⚡ Quarterly grid: {len(grid_pivot)} quarters")
print(f"   ISOs: {[c for c in grid_pivot.columns if c.startswith('queue_')]}")
print(grid_pivot.head())

⚡ Quarterly grid: 88 quarters
   ISOs: ['queue_CAISO', 'queue_MISO']
  quarter  queue_CAISO  queue_MISO
0  1999Q4          1.0         0.0
1  2000Q1          1.0         0.0
2  2000Q2          1.0         0.0
3  2000Q3          3.0         0.0
4  2000Q4          3.0         0.0


In [10]:
# Join grid to panel (all companies share same grid data per quarter)
df_panel = df_panel.merge(grid_pivot, on='quarter', how='left')
df_panel = df_panel.fillna(0)
print(f"✅ Joined grid: {len(df_panel)} rows")

✅ Joined grid: 231 rows


## Step 7: Label Promises (Target Variable)

In [11]:
# Map company_ticker to ticker
ticker_map = {
    'TICKER_MSFT': 'MSFT',
    'TICKER_GOOGL': 'GOOGL',
    'TICKER_AMZN': 'AMZN',
    'TICKER_META': 'META',
    'TICKER_NVDA': 'NVDA',
    'TICKER_DLR': 'DLR',
    'TICKER_EQIX': 'EQIX',
    'TICKER_AMT': 'AMT',
    'TICKER_PLD': 'PLD'
}

# Get promise quarters
df_promises['target_quarter'] = df_promises['target_date'].dt.to_period('Q').astype(str)
df_promises['ticker_clean'] = df_promises['company_ticker'].map(ticker_map)

# Create labels: promise_kept, promised_mw per (ticker, quarter)
promise_labels = df_promises[['ticker_clean', 'target_quarter', 'promise_kept', 'promised_mw']].copy()
promise_labels = promise_labels.rename(columns={
    'ticker_clean': 'ticker',
    'target_quarter': 'quarter'
})

print(f"🤝 Promise labels: {len(promise_labels)} rows")
print(promise_labels.head(10))

🤝 Promise labels: 34 rows
  ticker quarter  promise_kept  promised_mw
0   MSFT  2023Q4             1          100
1   MSFT  2024Q2             1          150
2   MSFT  2024Q4             1          200
3   MSFT  2026Q1             0          250
4  GOOGL  2023Q3             1          120
5  GOOGL  2024Q1             1          180
6  GOOGL  2024Q3             1          220
7  GOOGL  2026Q1             0          300
8   AMZN  2023Q2             1           90
9   AMZN  2024Q3             1          160


In [12]:
# Join labels to panel
df_panel = df_panel.merge(promise_labels, on=['ticker', 'quarter'], how='left')

# Fill missing labels (no promise that quarter) with NaN for promise_kept, 0 for promised_mw
df_panel['promised_mw'] = df_panel['promised_mw'].fillna(0)

# Create has_promise flag
df_panel['has_promise'] = df_panel['promise_kept'].notna().astype(int)

print(f"✅ Joined labels: {len(df_panel)} rows")
print(f"   Quarters with promises: {df_panel['has_promise'].sum()}")
print(f"   Promise kept rate: {df_panel['promise_kept'].mean():.1%}")

✅ Joined labels: 231 rows
   Quarters with promises: 25
   Promise kept rate: 76.0%


## Step 8: Final Cleanup and Save

In [13]:
# Reorder columns
col_order = [
    'ticker', 'quarter',
    # Financial features
    'q_avg_close', 'q_volatility', 'q_min_close', 'q_max_close',
    'q_avg_volume', 'q_total_volume',
    'q_avg_open', 'q_avg_high', 'q_avg_low',
    # Static ratios
    'market_cap', 'enterprise_value', 'revenue', 'ebitda',
    'profit_margin', 'operating_margin', 'return_on_assets', 'return_on_equity',
    'debt_to_equity', 'current_ratio', 'quick_ratio', 'dividend_yield',
    'beta', 'forward_pe', 'trailing_pe', 'price_to_book', 'price_to_sales',
    # Macro
    'Real GDP', 'Core CPI', 'Fed Funds Rate', 'Unemployment Rate', 
    '30Y Mortgage Rate', '10Y Treasury',
    # Grid
    'queue_CAISO', 'queue_ERCOT', 'queue_MISO', 'queue_PJM', 'queue_SPP',
    # Labels
    'promise_kept', 'promised_mw', 'has_promise'
]

# Keep only columns that exist
col_order = [c for c in col_order if c in df_panel.columns]
df_panel = df_panel[col_order]

print(f"📋 Final panel: {len(df_panel)} rows × {len(df_panel.columns)} columns")
print(f"\nColumns: {list(df_panel.columns)}")

📋 Final panel: 231 rows × 30 columns

Columns: ['ticker', 'quarter', 'market_cap', 'enterprise_value', 'revenue', 'ebitda', 'profit_margin', 'operating_margin', 'return_on_assets', 'return_on_equity', 'debt_to_equity', 'current_ratio', 'quick_ratio', 'dividend_yield', 'beta', 'forward_pe', 'trailing_pe', 'price_to_book', 'price_to_sales', 'Real GDP', 'Core CPI', 'Fed Funds Rate', 'Unemployment Rate', '30Y Mortgage Rate', '10Y Treasury', 'queue_CAISO', 'queue_MISO', 'promise_kept', 'promised_mw', 'has_promise']


In [14]:
# Display sample
print("\n📊 Sample data (first 5 rows):")
print(df_panel.head())

print("\n📊 Sample with promises:")
print(df_panel[df_panel['has_promise'] == 1].head(10))


📊 Sample data (first 5 rows):
  ticker quarter   market_cap  enterprise_value     revenue      ebitda  \
0    DLR  2021Q2  71328628736       89419341824  6080705024  2779584000   
1    DLR  2021Q3  71328628736       89419341824  6080705024  2779584000   
2    DLR  2021Q4  71328628736       89419341824  6080705024  2779584000   
3    DLR  2022Q1  71328628736       89419341824  6080705024  2779584000   
4    DLR  2022Q2  71328628736       89419341824  6080705024  2779584000   

   profit_margin  operating_margin  return_on_assets  return_on_equity  ...  \
0         0.2152           0.14147           0.01175           0.05469  ...   
1         0.2152           0.14147           0.01175           0.05469  ...   
2         0.2152           0.14147           0.01175           0.05469  ...   
3         0.2152           0.14147           0.01175           0.05469  ...   
4         0.2152           0.14147           0.01175           0.05469  ...   

   Core CPI  Fed Funds Rate  Unemployment R

In [15]:
# Summary statistics
print("\n📈 Summary Statistics:")
print(f"   Total rows: {len(df_panel)}")
print(f"   Unique companies: {df_panel['ticker'].nunique()}")
print(f"   Unique quarters: {df_panel['quarter'].nunique()}")
print(f"   Rows with promises: {df_panel['has_promise'].sum()}")
print(f"   Promise kept (where applicable): {df_panel['promise_kept'].dropna().mean():.1%}")

print("\n🔍 Missing values per column:")
missing = df_panel.isnull().sum()
print(missing[missing > 0])


📈 Summary Statistics:
   Total rows: 231
   Unique companies: 11
   Unique quarters: 21
   Rows with promises: 25
   Promise kept (where applicable): 76.0%

🔍 Missing values per column:
promise_kept    206
dtype: int64


In [16]:
# Save to CSV
output_path = 'data/processed/quarterly_panel.csv'
df_panel.to_csv(output_path, index=False)
print(f"✅ Saved to: {output_path}")

# Verify
df_check = pd.read_csv(output_path)
print(f"   Verified: {len(df_check)} rows × {len(df_check.columns)} columns")

✅ Saved to: data/processed/quarterly_panel.csv
   Verified: 231 rows × 30 columns


## Summary

✅ **Quarterly Panel Created**

- **Rows**: ~500 (Company × Quarter)
- **Columns**: 25+ features
- **Promise labels**: 34 quarters with promise events
- **Use case**: Regression, feature importance, cross-validation